In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

# Util

In [ ]:
import os
from pathlib import Path
from typing import Optional
from typing import List

def get_file_path(file_path: str) -> str:
    """
    파일 경로를 절대 경로로 변환하는 함수
    
    Args:
        file_path: 상대 경로 또는 절대 경로
    """
    # 파일 경로 확인 및 절대 경로로 변환
    file_path = Path(file_path)
    if not file_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        file_path = project_root / file_path
    
    if not file_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {file_path}")
    
    return str(file_path)


def table_to_markdown(table: List[List]) -> str:
    """
    표 데이터를 마크다운 테이블 형식으로 변환하는 헬퍼 함수
    
    Args:
        table: 2차원 리스트 형태의 표 데이터
    
    Returns:
        마크다운 테이블 문자열
    """
    if not table or len(table) == 0:
        return ""
    
    # 빈 셀을 빈 문자열로 변환
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).strip()
    
    # 표 데이터 정리
    cleaned_table = [[clean_cell(cell) for cell in row] for row in table]
    
    # 최대 컬럼 수 확인
    max_cols = max(len(row) for row in cleaned_table) if cleaned_table else 0
    
    # 모든 행을 동일한 컬럼 수로 맞춤
    normalized_table = []
    for row in cleaned_table:
        normalized_row = row + [""] * (max_cols - len(row))
        normalized_table.append(normalized_row)
    
    if not normalized_table:
        return ""
    
    markdown_lines = []
    
    # 헤더 행 (첫 번째 행)
    header = normalized_table[0]
    markdown_lines.append("| " + " | ".join(header) + " |")
    
    # 구분선
    markdown_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    
    # 데이터 행들
    for row in normalized_table[1:]:
        markdown_lines.append("| " + " | ".join(row) + " |")
    
    return "\n".join(markdown_lines)

# LLM 모델 생성

In [ ]:
# 문서가 변액일임펀드 설정/해지 지시서인지 확인하는 LLM 노드 생성

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")
LLM_TEMPERATURE=os.getenv("LLM_TEMPERATURE")

def create_llm_model():
    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:",
        temperature=LLM_TEMPERATURE,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY
    )
    return llm

# pdfplumber

In [ ]:
def extract_text_from_pdf_with_pdfplumber(pdf_path: str, password: str = None) -> str:
    """
    pdfplumber를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수
    
    pdfplumber는 PDF 파일을 텍스트 데이터로 추출하는 라이브러리로, 표, 이미지, 레이아웃 등을 잘 보존합니다.
    암호화된 PDF와 암호화되지 않은 PDF 모두 처리할 수 있습니다.
    """
    try:
        import pdfplumber
    except ImportError:
        raise ImportError(
            "PDF를 처리하기 위해 pdfplumber가 필요합니다.\n"
            "설치 명령: pip install pdfplumber"
        )
    
    markdown_parts = []
    
    try:
        pdf_path = get_file_path(pdf_path)
        # password가 있으면 암호화된 PDF로 처리, 없으면 암호화되지 않은 PDF로 처리
        pdf_kwargs = {"password": password} if password else {}
        
        with pdfplumber.open(pdf_path, **pdf_kwargs) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_content = []
                
                # 표 추출 (표가 있으면 먼저 표를 추출)
                tables = page.extract_tables()
                if tables:
                    for table_idx, table in enumerate(tables):
                        if table:
                            markdown_table = table_to_markdown(table)
                            if markdown_table:
                                page_content.append(markdown_table)
                                page_content.append("")  # 표 다음에 빈 줄 추가
                
                # 텍스트 추출
                text = page.extract_text()
                if text:
                    page_content.append(text)
                
                if page_content:
                    markdown_parts.append("\n".join(page_content))
        
        return "\n\n".join(markdown_parts) if markdown_parts else ""
        
    except Exception as e:
        # 암호화 관련 오류인지 확인
        error_msg = str(e).lower()
        if 'password' in error_msg or 'encrypted' in error_msg or 'incorrect password' in error_msg:
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        raise

# Docling

In [ ]:
from typing import Optional
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.accelerator_options import AcceleratorOptions

# ✅ 표 구조 복원에 유리한 권장 백엔드 (기본값이기도 함) :contentReference[oaicite:5]{index=5}
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend

try:
    from docling.document_converter import PdfBackendOptions
except ImportError:
    from docling.datamodel.base_models import PdfBackendOptions

from docling_core.types.doc.document import ContentLayer  # :contentReference[oaicite:3]{index=3}

def extract_text_from_pdf_with_docling3(pdf_path: str, password: str = None) -> str:
    pdf_path = get_file_path(pdf_path)

    def _run(do_cell_matching: bool) -> str:
        pipeline = PdfPipelineOptions(
            do_ocr=False,
            do_table_structure=True,
            do_picture_classification=False,
            do_picture_description=False,
            generate_page_images=True,
            images_scale=2.0,  # 레이아웃/테이블 크롭 품질에 도움될 수 있음 :contentReference[oaicite:6]{index=6}
        )

        # ✅ 표 구조 품질 우선
        pipeline.table_structure_options.mode = TableFormerMode.ACCURATE  # :contentReference[oaicite:7]{index=7}
        pipeline.table_structure_options.do_cell_matching = do_cell_matching  # :contentReference[oaicite:8]{index=8}

        # ✅ 표 구조 목적이면 force_backend_text는 끄는 쪽이 안전
        pipeline.force_backend_text = True  # :contentReference[oaicite:9]{index=9}

        pipeline.accelerator_options = AcceleratorOptions(device="cpu")

        backend_opts = PdfBackendOptions(password=password) if password else None

        converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline,
                    backend=DoclingParseV4DocumentBackend,
                    backend_options=backend_opts,
                )
            },
        )

        result = converter.convert(pdf_path)
        # md = result.document.export_to_markdown()
        md = result.document.export_to_markdown(
            included_content_layers={ContentLayer.BODY},
            page_break_placeholder="\n\n<!-- pagebreak -->\n\n",                 # (선택) 페이지 경계 표시 :contentReference[oaicite:5]{index=5}
        )

        # “테이블이 전혀 안 잡혔는지” 빠른 휴리스틱 (파이프 문자 기반)
        # 필요하면 여기서 result.document에서 TableItem 개수를 세는 방식으로 더 정확히 판단 가능 :contentReference[oaicite:10]{index=10}
        return md

    # 1차: 기본(셀 매칭 True)
    md = _run(do_cell_matching=True)
    if "|---" in md or "| ---" in md:
        print("do_cell_matching=True")
        return md

    # 2차: 셀 매칭 False (borderless/매칭 실패 케이스에 유리) :contentReference[oaicite:11]{index=11}
    md2 = _run(do_cell_matching=False)
    print("do_cell_matching=False")
    return md2


# document load

In [ ]:
# Document Loader - 파일 형식에 따라 적절한 함수 호출

from pathlib import Path
from typing import Union

def load_document(file_path: str, password: str = None, pdf_lib: str = 'pdfplumber') -> str:
    """
    파일 형식에 따라 적절한 텍스트 추출 함수를 호출하여 텍스트를 반환하는 통합 함수
    
    지원 형식:
    - PDF: .pdf 파일 (암호화된 PDF 지원)
    - Excel: .xlsx, .xls 파일
    
    Args:
        file_path: 문서 파일 경로 (상대 경로 또는 절대 경로)
        password: PDF 파일이 암호화된 경우 비밀번호 (선택사항)
    
    Returns:
        추출된 텍스트 문자열
        - PDF: 전체 텍스트
        - Excel: 모든 시트의 텍스트를 합친 문자열
    
    Raises:
        FileNotFoundError: 파일을 찾을 수 없을 때
        ValueError: 지원하지 않는 파일 형식일 때 또는 PDF 암호가 틀렸을 때
        ImportError: 필요한 라이브러리가 설치되지 않았을 때
    """
    file_path_obj = Path(file_path)
    file_ext = file_path_obj.suffix.lower()
    
    # 파일 형식에 따라 적절한 함수 호출
    if file_ext == '.pdf':
        # PDF 파일 처리 (암호 전달)
        if pdf_lib == 'pdfplumber':
            return extract_text_from_pdf_with_pdfplumber(file_path, password)
        elif pdf_lib == 'docling':
            return extract_text_from_pdf_with_docling3(file_path, password)
    
    # elif file_ext in ['.xlsx', '.xls']:
    #     # Excel 파일 처리 - 모든 시트의 텍스트를 하나의 문자열로 반환
    #     return get_all_sheets_text(file_path)
    
    else:
        raise ValueError(
            f"지원하지 않는 파일 형식입니다: {file_ext}\n"
            f"지원 형식: .pdf, .xlsx, .xls"
        )

# LLM 문서 정리

In [ ]:
# system 프롬프트
_SYSTEM_PROMPT = """당신은 자산운용사에서 해외거래체결 확인을 담당하는 오퍼레이터 입니다.
    당신의 역할은 시스템에 자동 피딩된 해외거래체결내역 정보와 브로커가 메일로 보내온 해외거래체결내역 확인서를 비교하기 위하여, 
    브로커가 보낸 해외거래체결내역 확인서 파일에서 비교 대상 데이터를 수집하고 정리하는 역할입니다.
"""

In [ ]:
# system 프롬프트
_SYSTEM_PROMPT2 = """
당신은 자산운용사의 해외주식 거래 체결내역 검증(Trade Confirmation Reconciliation) 담당 오퍼레이터입니다.

목표:
- 시스템에 자동 피딩된 해외거래 체결정보와, 브로커가 이메일로 보낸 “해외거래체결 확인서(PDF)”의 내용을 거래 단위로 정확히 대조할 수 있도록,
  PDF에서 비교 대상 데이터를 추출·정리하여 표 형태로 제공합니다.

원칙:
- 원문에 없는 내용은 절대 추정/생성하지 않습니다.
- 요약/통합하지 않고, 거래(Trade) 단위로 모두 분리해 정리합니다.
- docling 추출본을 주 분석 근거로 사용하되, 단어/코드/숫자의 정확성은 pdfplumber 추출본으로 교차검증하여 더 정확한 값을 선택합니다.
- PDF 추출 특성(공백/줄바꿈/컬럼 병합/중복/깨짐)으로 인한 오류를 탐지하고, 근거 기반으로만 수정합니다.
- 출력은 반드시 Markdown으로만 작성합니다.
"""

In [ ]:
_HUMAN_PROMPT = """
아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber와 docling 라이브러리를 사용하여 추출한 data입니다.
아래의 주식 해외거래체결내역 확인서 추출 data를 시스템에 자동 피딩된 정보와 비교 자료로 사용할 수 있도록 정리하세요.
지침에 따라 정리한 내용을 markdown 형식으로 작성하세요.

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - docling 라이브러리 사용 ###
{document_text_docling}    

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - pdfplumber 라이브러리 사용 ###
{document_text_pdfplumber}

** 반드시 지켜야 할 중요 지침 **
1. docling 추출 data를 기반으로 전체 내용을 분석하세요.
2. 통합 또는 요약하지 말고, 거래 단위로 data를 정리하세요.
3. 각 단어와 코드, 숫자 데이터들을 pdfplumber 추출 data와 비교하여 보다 정확한 data를 선택하세요.
4. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
5. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
6. PDF에서 추출한 data의 특성상 인접한 컬럼의 데이터가 중복되거나, 인접한 컬럼으로 병합되는 오류가 발생할 수 있습니다. 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.
7. 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.
8. 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.
9. 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.
10. PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
11. 종목명(stock name, security name, name)에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
12. 종목명(stock name, security name, name)에서 약어를 사용하는지 판단하여 약어를 유지하세요.  
13. 원문을 번역하지 말고 원문 그대로 출력하세요.
"""

In [ ]:
_HUMAN_PROMPT2 = """
아래 입력은 브로커가 보낸 “주식 해외거래체결내역 확인서(Trade Confirmation)” PDF에서 추출한 텍스트입니다.
- docling 추출본: 구조 파악/전체 분석의 1차 기준
- pdfplumber 추출본: 단어/코드/숫자 정확도 교정 및 누락 보완의 2차 기준

당신의 작업 목표:
- 아래 추출 데이터를 “시스템 자동 피딩 정보”와 1:1 비교할 수 있도록, 거래(Trade) 단위로 모든 비교 대상 필드를 정리해 Markdown으로 출력하세요.

입력 데이터
### [A] docling 추출 텍스트 (1차 기준)
{document_text_docling}

### [B] pdfplumber 추출 텍스트 (정확도 교정/누락 보완)
{document_text_pdfplumber}


========================
필수 작업 지침 (반드시 준수)
========================
1) 분석 기준
- 전체 구조/거래 구분/섹션 구분은 [A] docling을 기준으로 파악하세요.
- 단어/코드/숫자(예: 수량, 단가, 금액, 통화, 식별코드 등)는 [B] pdfplumber와 대조하여 더 정확한 값을 선택하세요.

2) 거래 단위 정리 (요약/통합 금지)
- 문서에 거래가 N건이면 결과도 N개의 거래 레코드로 분리하세요.
- 여러 거래를 합쳐서 한 줄로 요약하지 마세요.

3) 값 선택 규칙 (충돌 처리)
- docling과 pdfplumber 값이 다르면:
  a) 숫자/코드/티커/ISIN/계좌/참조번호 등 “식별/정량” 값은 pdfplumber 우선
  b) 문장/레이블/항목명/섹션명은 docling 우선
- 어느 쪽이 맞는지 판단 불가하면 임의로 결정하지 말고, 해당 셀에 그대로 두되 **CONFLICT 표시**를 남기세요.
  예) `CONFLICT: docling=... | pdfplumber=...`

4) 누락 점검 및 보완
- 거래별로 필수 비교 필드가 비어있지 않은지 점검하세요.
- 한쪽 추출본에만 존재하는 값이 있으면 누락 없이 채우세요.
- 그래도 찾을 수 없으면 `MISSING`으로 표기하세요(추정 금지).

5) 금액/합계 검증
- 문서에 합계/총액/Total/Sum 같은 값이 있다면:
  - 거래별 합계를 계산해 문서 표기 합계와 일치하는지 검증 결과를 표시하세요.
  - 불일치 시 어떤 항목이 차이나는지(수량/단가/수수료/세금/환율 등) 가능한 범위에서 원문 근거로 설명하세요.
- 문서에 합계가 없으면 “합계 표기 없음”으로 명시하세요.

6) 컬럼 병합/중복 오류 교정
- PDF 추출 특성상 인접 컬럼이 합쳐지거나 값이 옆 컬럼에 중복되는 경우가 있습니다.
- 결과 테이블에서 컬럼 간 값이 중복/병합된 흔적이 있으면 [B] pdfplumber로 교정하세요.
- 교정은 “원문에 존재하는 토큰”을 재배치하는 수준에서만 수행(새 값 생성 금지).

7) 원문 보존 (삭제/요약/새로작성 금지)
- 단어/숫자/코드를 임의로 제거하거나 요약하거나 새로 만들지 마세요.
- 번역하지 말고 원문 그대로 출력하세요(영문/약어/코드 유지).

8) 공백/줄바꿈 교정 (의미 기반, 최소 수정)
- 공백/띄어쓰기/줄바꿈 오류는 “의미와 맥락상 명백한 경우”에만 최소한으로 수정하세요.
- 특히 종목명(Security/Stock name)에서 줄바꿈/공백 오류가 자주 발생하므로 우선 점검하세요.
- 종목명의 약어는 약어로 판단되면 유지하세요(임의로 풀어쓰지 않음).


========================
출력 형식 (Markdown 고정)
========================
아래 순서와 섹션명을 그대로 사용하세요.

## 1) 문서 식별 정보 (있으면 추출, 없으면 MISSING)
- Broker / Counterparty:
- Document Type / Title:
- Trade Date / Confirmation Date:
- Account / Portfolio:
- Reference No / Confirmation No:
- Currency:
- 기타 식별 정보:

## 2) 거래(Trade) 단위 정리 테이블
- 거래 1건 = 테이블 1행 (절대 합치지 말 것)
- 가능한 한 많은 비교 필드를 컬럼으로 구성
- 추천 컬럼(문서에 있는 것만 채우고 없으면 MISSING):
  - Trade No/Ref
  - Buy/Sell
  - Security Name
  - Ticker/ISIN/SEDOL (있는 식별코드 모두)
  - Market/Exchange
  - Trade Date
  - Settlement Date
  - Quantity
  - Price
  - Gross Amount
  - Fees/Commission
  - Taxes
  - Net Amount
  - Currency
  - FX Rate (있으면)
  - 기타(원문에 존재하는 추가 필드)
  - Source Note (선택값 근거: docling/pdfplumber/CONFLICT)

## 3) 누락/충돌/이상치 점검 결과
- MISSING 항목 목록 (거래별)
- CONFLICT 항목 목록 (거래별, docling vs pdfplumber 원문값 병기)
- 컬럼 병합/중복 교정 내역(발견 시)

## 4) 합계 검증 결과
- 문서 합계 표기: (있으면 값 그대로 / 없으면 “합계 표기 없음”)
- 계산 합계(가능한 경우):
- 일치 여부: OK / MISMATCH
- MISMATCH이면 차이 원인 후보(원문 근거 기반)

주의:
- 결과는 Markdown만 출력하세요.
- 표/항목에 없는 내용은 절대 추정하지 마세요.

"""

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def convert_document_text_to_markdown(document_text: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
    아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber 라이브러리를 사용하여 추출한 data입니다.
    추출 data는 table을 markdown 형식으로 변환하여 추출한 data와 텍스트 형식으로 추출한 data로 구성되어 있습니다.
    지침에 따라 추출 data를 정리하세요.
    
    ### 변액일임펀드 설정/해지 지시서 내용 ###
    {document_text}

    ** 반드시 지켜야 할 중요 지침 **
    1. 전체 내용을 분석하세요.
    2. 전체 내용을 markdown 형식으로 수정하세요.
    3. markdown table 코드를 오류가 없는 정상적인 코드로 수정하세요.    
    4. 추측과 예상 또는 설명 등의 첨언은 하지 말고 추출 data 내용만 출력하세요.
    5. 중복되는 내용은 제거하세요.
    6. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
    7. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
    8. 정리 결과에서 종목명과 코드가 정확한지 확인하고 오류가 있으면 수정하세요.
    9. 정리 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.    
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def convert_document_text_to_markdown2(document_text_docling: str, document_text_pdfplumber: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
    아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber와 docling 라이브러리를 사용하여 추출한 data입니다.
    추출 data는 table을 markdown 형식으로 변환하여 추출한 data와 텍스트 형식으로 추출한 data로 구성되어 있습니다.
    지침에 따라 추출 data를 정리하세요.
    
    ### 변액일임펀드 설정/해지 지시서 내용 - docling 라이브러리 사용 ###
    {document_text_docling}    

    ### 변액일임펀드 설정/해지 지시서 내용 - pdfplumber 라이브러리 사용 ###
    {document_text_pdfplumber}

    ** 반드시 지켜야 할 중요 지침 **
    1. docling 추출 data를 기반으로 전체 내용을 분석하세요.
    2. 각 단어와 숫자 데이터들을 pdfplumber 추출 data와 비교하여 보다 정확한 data를 선택하세요.
    3. 전체 내용을 markdown 형식으로 수정하세요.
    4. markdown table 코드를 오류가 없는 정상적인 코드로 수정하세요.
    5. 중복되는 내용은 제거하세요.
    6. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
    7. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
    8. 정리 결과의 테이블에서 컬럼 데이터가 인접한 컬럼에 중복되어 작성되어 있는지 확인하세요. 중복되어 작성되어 있으면 제거하세요.
    9. 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.
    10. 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.
    11. 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.
    12. PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
    13. 종목명(stock name, security name) 지침
        13.1. 종목명(stock name, security name, name)은 시스템 표준이므로 단어의 철자를 추가하거나 변경하지말고 그대로 유지하세요.
        13.2. **종목명(stock name, security name, name)은 시스템 표준이므로 축약된 단어가 사용될 수 있습니다. 이를 복원하면 시스템에서 오류가 발생할 수 있으므로 이를 복원하지 마세요.**
        13.3. **단어의 띄어쓰기, 공백, 줄바꿈 오류만 수정하세요.**
        13.4. 종목명(stock name, security name, name)의 의미를 분석하고 맥락을 통해 단어의 띄어쓰기 및 공백 오류가 있는지 확인하고 오류가 있으면 수정하세요.
        13.5. 종목명(stock name, security name, name)의 의미를 분석하여 맥락을 통해 단어가 중간에서 줄바꿈으로 잘려진 사실이 확인되면 수정하세요.           
    14. 원문을 번역하지 말고 원문 그대로 출력하세요.
    15. 추측과 예상 또는 설명 등의 첨언은 하지 말고 추출 data 내용만 출력하세요.
    16. 정리 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.    
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def convert_document_text_to_markdown3(document_text_docling: str, document_text_pdfplumber: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber와 docling 라이브러리를 사용하여 추출한 data입니다.
아래의 주식 해외거래체결내역 확인서 추출 data를 시스템에 자동 피딩된 정보와 비교 자료로 사용할 수 있도록 정리하세요.
지침에 따라 정리한 내용을 markdown 형식으로 작성하세요.

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - docling 라이브러리 사용 ###
{document_text_docling}    

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - pdfplumber 라이브러리 사용 ###
{document_text_pdfplumber}

** 반드시 지켜야 할 중요 지침 **
1. docling 추출 data를 기반으로 전체 내용을 분석하세요.
2. 통합 또는 요약하지 말고, 거래 단위로 data를 정리하세요.
3. 각 단어와 코드, 숫자 데이터들을 pdfplumber 추출 data와 비교하여 보다 정확한 data를 선택하세요.
4. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
5. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
6. PDF에서 추출한 data의 특성상 인접한 컬럼의 데이터가 중복되거나, 인접한 컬럼으로 병합되는 오류가 발생할 수 있습니다. 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.
7. 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.
8. 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.
9. 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.
10. PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
11. 종목명(stock name, security name, name)에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
12. 종목명(stock name, security name, name)에서 약어를 사용하는지 판단하여 약어를 유지하세요.  
13. 원문을 번역하지 말고 원문 그대로 출력하세요.
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

# LLM 문서 검증

In [ ]:
_SYSTEM_PROMPT_VALID = """
당신은 자산운용사의 해외주식 거래체결 확인(Trade Confirmation Reconciliation) 담당 오퍼레이터입니다.

업무 목적:
- 시스템에 자동 피딩된 해외거래 체결정보와 브로커가 보낸 거래체결 확인서(PDF) 내용을 비교할 수 있도록,
  확인서에서 필요한 거래 단위 데이터를 정확하게 정리/검수합니다.

업무 범위:
- 입력으로 제공되는 Markdown 정리본(document_markdown)을 “검수 대상”으로 삼고,
  PDF에서 추출된 원문(docling, pdfplumber)과 대조하여 오류를 수정합니다.

원칙:
- 원문(PDF 추출 텍스트)에 없는 정보는 추정/생성하지 않습니다.
- 번역하지 않고 원문 언어/표기를 유지합니다.
- docling은 구조/문맥 파악에 우선 사용하고, 숫자·코드·철자 등 정밀 값은 pdfplumber로 교차검증합니다.
- 최종 출력은 시스템 자동 피딩 데이터와 비교에 필요한 “해외거래 체결내역 핵심 필드”만 Markdown으로 제공합니다.

"""

In [ ]:
_HUMAN_PROMPT_VALID = """"
markdown으로 정리한 주식 해외거래체결내역 확인서 내용을 검수하세요.

pdfplumber와 docling 라이브러리를 사용하여 주식 해외거래체결내역 확인서 PDF 파일에서 추출한 원문 data를 참고하여 아래의 지침에 따라 검수하세요.

### 주식 해외거래체결내역 확인서 markdown 정리 내용 - 검수 대상 ###
{document_markdown}

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - docling 라이브러리 사용 ###
{document_text_docling}    

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - pdfplumber 라이브러리 사용 ###
{document_text_pdfplumber}

** 반드시 지켜야 할 중요 지침 **
1. 누락된 데이터가 있는지 PDF 파일 내용 추출 data와 비교하여 확인하세요. 누락된 데이터가 있으면 추가하세요.
2. 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
3. 테이블의 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 PDF 파일 내용 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.    
4. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
5. 종목명(stock name, security name, name)에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
6. 종목명(stock name, security name, name)을 PDF 파일 내용 추출 data와 비교하여 단어의 철자가 다르거나 단어가 추가된 경우 원문을 유지하도록 수정하세요.
7. 종목명(stock name, security name, name)을 PDF 파일 내용 추출 data와 비교하여 약어를 풀로 표기한 경우 약어로 수정하세요.  
8. 검수 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.
9. 시스템 자동 피딩 데이터와 비교하는 작업에 필요한 해외거래체결내역 정보만 출력하세요.    
"""

In [ ]:
_HUMAN_PROMPT_VALID2 = """
아래 입력은 브로커가 보낸 “주식 해외거래체결내역 확인서”를 정리한 Markdown(검수 대상)과,
같은 PDF에서 추출한 원문 텍스트(docling / pdfplumber)입니다.

당신의 목표:
- 검수 대상 Markdown({document_markdown})을 원문 추출 데이터([A], [B])와 대조하여,
  누락/오류/중복/병합/공백-줄바꿈 문제를 수정한 “최종 검수본 Markdown”을 출력하세요.
- 최종 출력에는 시스템 자동 피딩 데이터와 비교에 필요한 해외거래 체결내역 정보만 포함하세요.

입력
### [검수 대상] Markdown 정리본
{document_markdown}

### [A] PDF 원문 추출 텍스트 - docling (구조/문맥 1차 기준)
{document_text_docling}

### [B] PDF 원문 추출 텍스트 - pdfplumber (정밀 값 2차 기준)
{document_text_pdfplumber}


========================
검수 및 수정 규칙 (반드시 준수)
========================
1) 누락 점검 및 추가
- 검수 대상 Markdown을 [A], [B]와 비교하여 거래(Trade) 단위 필드가 누락되었는지 확인하세요.
- 누락된 값이 원문에 존재하면 반드시 추가하세요.
- 원문에서 찾을 수 없으면 추정하지 말고 `MISSING`으로 표기하세요.

2) 금액/합계 검증
- 문서에 합계/총액/Total/Sum이 존재하면:
  - 거래별 금액(가능한 범위)을 합산하여 문서 합계와 일치하는지 검증 결과를 표시하세요.
  - 불일치하면 `MISMATCH`로 표시하고, 어떤 항목(수량/단가/수수료/세금/환율/순액 등)이 관련되는지
    원문 근거가 있는 범위에서만 적으세요.
- 문서에 합계 표기가 없으면 “합계 표기 없음”을 명시하세요.

3) 테이블 컬럼 병합/중복 교정
- PDF 추출 특성상 인접 컬럼 값이 합쳐지거나 다른 컬럼에 중복될 수 있습니다.
- 검수본 테이블에서 컬럼 간 값이 병합/중복된 흔적이 있으면 [B]를 우선 근거로 올바른 컬럼에 재배치하세요.
- 재배치는 “원문에 존재하는 토큰”을 옮기는 수준에서만 수행(새 값 생성 금지).

4) 공백/띄어쓰기/줄바꿈 교정(의미 기반, 최소 수정)
- 의미와 맥락상 명백한 경우에만 공백/줄바꿈을 최소한으로 수정하세요.
- 특히 종목명(Security/Stock name)에서 줄바꿈/공백 오류가 잦으므로 우선 점검하세요.

5) 종목명 정합성(철자/단어 추가/약어 유지)
- 종목명은 [A], [B]와 비교하여 철자가 다르거나 불필요한 단어가 추가된 경우 원문 표기를 유지하도록 수정하세요.
- 종목명이 풀어쓰기(확장)로 바뀐 경우, 원문이 약어라면 약어 형태로 되돌리세요.
  (단, 원문에 실제로 약어가 존재할 때만 수정)

6) 교차검증 우선순위
- 구조/섹션/레이블/항목명: [A] docling 우선
- 숫자/코드/티커/ISIN/참조번호/계좌/수량/단가/금액: [B] pdfplumber 우선
- 판단 불가하면 임의로 결정하지 말고 `CONFLICT: docling=... | pdfplumber=...`로 남기세요.

7) 최종 검증
- 수정 후 전체 문서를 다시 점검하여,
  (a) 누락, (b) 합계 오류, (c) 컬럼 병합/중복, (d) 종목명 오류가 남아있지 않게 하세요.

8) 출력 제한 (중요)
- 최종 출력에는 “시스템 자동 피딩 데이터와 비교에 필요한 해외거래체결내역 정보”만 포함하세요.
- 불필요한 설명/요약/번역/사족은 출력하지 마세요.


========================
최종 출력 형식 (Markdown 고정)
========================
아래 섹션/순서를 그대로 사용하여 “수정 반영된 최종 검수본”만 출력하세요.

## 1) 문서 식별 정보 (원문에 있는 것만)
- Broker / Counterparty:
- Reference No / Confirmation No:
- Trade Date:
- Settlement Date:
- Account / Portfolio:
- Currency:
- 기타 식별정보:

## 2) 거래(Trade) 단위 체결내역 테이블
- 1 거래 = 1 행
- 컬럼은 원문에 존재하는 항목만 채우고, 없으면 MISSING
- 권장 컬럼:
  - Trade Ref/No
  - Buy/Sell
  - Security Name
  - Ticker / ISIN / SEDOL (존재하는 식별코드 모두)
  - Market/Exchange
  - Quantity
  - Price
  - Gross Amount
  - Fees/Commission
  - Taxes
  - Net Amount
  - Currency
  - FX Rate (있으면)
  - Notes (CONFLICT/MISSING 등)

## 3) 합계 검증 결과
- 문서 합계 표기:
- 계산 합계(가능하면):
- 검증 결과: OK / MISMATCH / 합계 표기 없음
- MISMATCH 상세(원문 근거가 있는 경우만):

## 4) 수정/보완 로그(간단히)
- 누락 추가:
- 컬럼 병합/중복 수정:
- 종목명 수정:
- 공백/줄바꿈 수정:
- CONFLICT 잔존(있으면):

"""

In [ ]:
def validation_markdown_document_with_llm(document_text_docling: str, document_text_pdfplumber: str, document_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
markdown으로 정리한 주식 해외거래체결내역 확인서 내용을 검수하세요.

pdfplumber와 docling 라이브러리를 사용하여 주식 해외거래체결내역 확인서 PDF 파일에서 추출한 원문 data를 참고하여 아래의 지침에 따라 검수하세요.

### 주식 해외거래체결내역 확인서 markdown 정리 내용 - 검수 대상 ###
{document_markdown}

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - docling 라이브러리 사용 ###
{document_text_docling}    

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - pdfplumber 라이브러리 사용 ###
{document_text_pdfplumber}

** 반드시 지켜야 할 중요 지침 **
1. 누락된 데이터가 있는지 PDF 파일 내용 추출 data와 비교하여 확인하세요. 누락된 데이터가 있으면 추가하세요.
2. 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
3. 테이블의 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 PDF 파일 내용 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.    
4. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
5. 종목명(stock name, security name, name)에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
6. 종목명(stock name, security name, name)을 PDF 파일 내용 추출 data와 비교하여 단어의 철자가 다르거나 단어가 추가된 경우 원문을 유지하도록 수정하세요.
7. 종목명(stock name, security name, name)을 PDF 파일 내용 추출 data와 비교하여 약어를 풀로 표기한 경우 약어로 수정하세요.  
8. 검수 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.
9. 시스템 자동 피딩 데이터와 비교하는 작업에 필요한 해외거래체결내역 정보만 출력하세요.
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

# LLM 데이터 추출

In [ ]:
def oversea_data_gethring_llm(document_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
아래는 브로커가 보내온 주식 해외거래체결내역 확인서 입니다.

당신의 작업 목표
- 시스템에 자동 피딩된 정보와 비교하기 위한 data를 지침에 따라 아래의 주식 해외거래체결내역 확인서에서 추출하세요.

### 주식 해외거래체결내역 확인서 ###
{document_markdown}

========================
필수 작업 지침 (반드시 준수)
========================
1) 거래 단위 정리 (요약/통합 금지)
- 문서에 거래가 N건이면 결과도 N개의 거래 레코드로 분리하세요.
- 여러 거래를 합쳐서 한 줄로 요약하지 마세요.

2) 누락 점검 및 보완
- 거래별로 필수 비교 필드가 비어있지 않은지 점검하세요.
- 한쪽 추출본에만 존재하는 값이 있으면 누락 없이 채우세요.
- 그래도 찾을 수 없으면 `MISSING`으로 표기하세요(추정 금지).

3) 원문 보존 (삭제/요약/새로작성 금지)
- 단어/숫자/코드를 임의로 제거하거나 요약하거나 새로 만들지 마세요.
- 번역하지 말고 원문 그대로 출력하세요(영문/약어/코드 유지).

4) Fund Code 작성 검증
- 아래의 탐색 방법과 예제를 참고하여 Fund Code가 올바르게 작성되었는지 확인하세요.
- Account Name 우측에 표시되며, 괄호에 묶여 표시되는 경우도 있음    
  예) Account Name : 삼성자산운용㈜(2ETFN2), Account Code : 2ETFN2
  예) Account Name : 한화생명보험(변액_삼성_주식)462018, Account Code : 462018

5) Fund Name 작성 검증
- Fund Code와 Fund Name 값이 동일하면 Fund Name 값을 MISSING 표기

6) Excuting Broker 작성 검증
- 아래의 탐색 방법과 예제를 참고하여 Excuting Broker가 올바르게 작성되었는지 확인하세요.
  a-타입) Trading Broker 필드
  b-타입) 표기 형식: [거래 확인서를 발행/연관된 브로커 또는 기관의 SWIFT/BIC 코드(Institution BIC)] + '(DTC' + [DTC에서 결제를 수행하는 참가자 번호] + ')'
         예) 'GBSLGB2LXXX' + '(DTC' + '2711)' = GBSLGB2LXXX(DTC2711)

7) Clearing Broker 작성 검증
- 아래의 탐색 방법과 예제를 참고하여 Clearing Broker가 올바르게 작성되었는지 확인하세요.
  a-타입) Clearing Broker 필드
  b-타입) 표기 형식: [Custodian(수탁/커스터디 은행)의 SWIFT/BIC 코드(Custodian BIC)] + '(DTC' + [DTC에서 결제를 수행하는 참가자 번호] + ')'
         예) 'IRVTUS3NSTC' + '(DTC' + '2711)' = IRVTUS3NSTC(DTC2711)

8) Sec Account 작성 검증
- 아래의 탐색 방법과 예제를 참고하여 Sec Account가 올바르게 작성되었는지 확인하세요.
- Account 필드 또는 CSD(Central Securities Depository, 중앙예탁결제기관) ID
- Sec Account와 Fund Code 값이 동일하면 Sec Account 값을 MISSING 표기

9) Clearing Agent ID 작성 검증
- 아래의 탐색 방법과 예제를 참고하여 Clearing Agent ID가 올바르게 작성되었는지 확인하세요.
- Participant ID 필드값의 '/' 기준 후열 문자
  예) DTCYID/0019 -> 0019

10) 최종 검증
- 수정 후 전체 문서를 다시 점검하여,
  (a) 누락, (b) 합계 오류, (c) 컬럼 병합/중복, (d) 종목명 오류가 남아있지 않게 하세요.

11) 출력 제한 (중요)
- 최종 출력에는 “시스템 자동 피딩 데이터와 비교에 필요한 해외거래체결내역 정보”만 포함하세요.
- 불필요한 설명/요약/번역/사족은 출력하지 마세요.

========================
출력 형식 (Markdown 고정)
========================
1) 거래(Trade) 단위 정리 테이블
- 거래 1건 = 테이블 1행 (절대 합치지 말 것)
- 가능한 한 많은 비교 필드를 컬럼으로 구성
- 필수 컬럼(문서에 있는 것만 채우고 없으면 MISSING):
  - Trade Date
  - Fund Code
  - Fund Name/Account Name
  - Ticker (Ticker값 + [국가코드], 예: SPY US)
  - ISIN
  - Security Name
  - Settlement Date
  - Buy/Sell
  - Currency
  - Excuted Qty
  - Deal Price
  - Gross Amount
  - Commission
  - Taxes
  - Other Charges
  - Net Settlement AMT
  - Executing Broker
  - Clearing Broker
  - Settlement Location (PSET)
  - Sec Account
  - Clearing Agent ID
- 기타 컬럼(필수 컬럼을 제외한 원문에 존재하는 추가 필드)

    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

# 테스트

In [ ]:
from IPython.display import Markdown, display

def display_markdown(response):
    # LLM 응답을 마크다운 형식으로 보기 좋게 표시
    if 'response' in locals():
        display(Markdown(response.content))
        
        # 추가 정보 (토큰 사용량 등)를 표시
        if hasattr(response, 'response_metadata') and response.response_metadata:
            metadata = response.response_metadata
            if 'token_usage' in metadata:
                print("\n---")
                print("**토큰 사용량:**")
                print(f"- 입력 토큰: {metadata['token_usage'].get('prompt_tokens', 'N/A')}")
                print(f"- 출력 토큰: {metadata['token_usage'].get('completion_tokens', 'N/A')}")
                print(f"- 총 토큰: {metadata['token_usage'].get('total_tokens', 'N/A')}")
    else:
        print("⚠️ 'response' 변수를 찾을 수 없습니다. 먼저 LLM을 호출해주세요.")

In [109]:
# text 추출 대상 파일 설정
# 암호 추출

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/대신.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/메리츠.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/미래.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/삼성.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/신한.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/유진.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/키움.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/하나.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/한국.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/DB.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/JP Morgan.pdf"
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/KB.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/NH.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/SB(Bernstein).pdf"

_password = None
# _password = '345678'

In [110]:
# pdfplumber 사용 text 추출

document_text_pdfplumber = load_document(_document_file_path, _password, pdf_lib='pdfplumber')
# print(document_text_pdfplumber)

In [111]:
# docling 사용 text 추출

document_text_docling = load_document(_document_file_path, _password, pdf_lib='docling')
# print(document_text_docling)

2026-01-08 18:36:04,186 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-08 18:36:04,190 - INFO - Going to convert document batch...
2026-01-08 18:36:04,190 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 3a85fbd3fac1234bc9da7957f69a8ff6
2026-01-08 18:36:04,190 - INFO - Accelerator device: 'cpu'
2026-01-08 18:36:04,757 - INFO - Accelerator device: 'cpu'
2026-01-08 18:36:05,314 - INFO - Processing document KB.pdf
2026-01-08 18:36:21,685 - INFO - Finished converting document KB.pdf in 17.50 sec.


do_cell_matching=True


In [112]:
res_markdown = convert_document_text_to_markdown3(document_text_docling, document_text_pdfplumber)
# display_markdown(res_markdown)

2026-01-08 18:37:59,037 - INFO - HTTP Request: POST http://localhost:3900/v1/chat/completions "HTTP/1.1 200 OK"


# 재 검증

In [ ]:
valid_markdown = validation_markdown_document_with_llm(document_text_docling, document_text_pdfplumber, res_markdown)
# display_markdown(valid_markdown)

# 데이터 추출

In [ ]:
data_markdown = oversea_data_gethring_llm(res_markdown)
display_markdown(data_markdown)